In [3]:
# importing all the libraries
import pandas as pd
import numpy as np
import  difflib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Data Collection and Preprocessing

In [4]:
# loading the data from the file
movies=pd.read_csv("Dataset\movies_data.csv")
movies

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811
...,...,...,...,...,...,...,...,...,...
9995,10196,The Last Airbender,"Action,Adventure,Fantasy",en,"The story follows the adventures of Aang, a yo...",98.322,2010-06-30,4.7,3347
9996,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",en,The sharks take bite out of the East Coast whe...,12.490,2015-07-22,4.7,417
9997,13995,Captain America,"Action,Science Fiction,War",en,"During World War II, a brave, patriotic Americ...",18.333,1990-12-14,4.6,332
9998,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",en,A man named Farmer sets out to rescue his kidn...,15.159,2007-11-29,4.7,668


In [5]:
# printing first 5 rows from the dataframe
movies.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


In [6]:
# number of Rows and Columns in data frame
movies.shape

(10000, 9)

In [7]:
# description about the data
movies.describe()

,id,popularity,vote_average,vote_count
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,161243.505000,34.697267,6.621150,1547.309400
std,211422.046043,211.684175,0.766231,2648.295789
min,5.000000,0.600000,4.600000,200.000000
25%,10127.750000,9.154750,6.100000,315.000000
50%,30002.500000,13.637500,6.600000,583.500000
75%,310133.500000,25.651250,7.200000,1460.000000
max,934761.000000,10436.917000,8.700000,31917.000000


In [8]:
# information about the data
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10000 non-null  int64  
 1   title              10000 non-null  object 
 2   genre              9997 non-null   object 
 3   original_language  10000 non-null  object 
 4   overview           9987 non-null   object 
 5   popularity         10000 non-null  float64
 6   release_date       10000 non-null  object 
 7   vote_average       10000 non-null  float64
 8   vote_count         10000 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB


In [9]:
# checking the null values
movies.isnull().sum()

id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

In [10]:
# cleaning the null values
movies=movies.fillna("")

In [11]:
# again checking the null values
movies.isnull().sum()

id                   0
title                0
genre                0
original_language    0
overview             0
popularity           0
release_date         0
vote_average         0
vote_count           0
dtype: int64

In [12]:
# name of each cloumns in dataframe
movies.columns

Index(['id', 'title', 'genre', 'original_language', 'overview', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

In [13]:
# Selecting the relevent features for recommendation

movies=movies[["title","overview","genre"]]
movies

,title,overview,genre
0,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,"Drama,Crime"
1,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...","Comedy,Drama,Romance"
2,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama,Crime"
3,Schindler's List,The true story of how businessman Oskar Schind...,"Drama,History,War"
4,The Godfather: Part II,In the continuing saga of the Corleone crime f...,"Drama,Crime"
...,...,...,...
9995,The Last Airbender,"The story follows the adventures of Aang, a yo...","Action,Adventure,Fantasy"
9996,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...,"Action,TV Movie,Science Fiction,Comedy,Adventure"
9997,Captain America,"During World War II, a brave, patriotic Americ...","Action,Science Fiction,War"
9998,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...,"Adventure,Fantasy,Action,Drama"


In [14]:
# Combining the two features with title of tags

movies["tags"] = movies["overview"] + movies["genre"] +  movies["title"]
movies

,title,overview,genre,tags
0,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,"Drama,Crime",Framed in the 1940s for the double murder of h...
1,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...","Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second..."
2,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama,Crime","Spanning the years 1945 to 1955, a chronicle o..."
3,Schindler's List,The true story of how businessman Oskar Schind...,"Drama,History,War",The true story of how businessman Oskar Schind...
4,The Godfather: Part II,In the continuing saga of the Corleone crime f...,"Drama,Crime",In the continuing saga of the Corleone crime f...
...,...,...,...,...
9995,The Last Airbender,"The story follows the adventures of Aang, a yo...","Action,Adventure,Fantasy","The story follows the adventures of Aang, a yo..."
9996,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...,"Action,TV Movie,Science Fiction,Comedy,Adventure",The sharks take bite out of the East Coast whe...
9997,Captain America,"During World War II, a brave, patriotic Americ...","Action,Science Fiction,War","During World War II, a brave, patriotic Americ..."
9998,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...,"Adventure,Fantasy,Action,Drama",A man named Farmer sets out to rescue his kidn...


In [15]:
# Removing these two features after forming the tags title

new_data = movies.drop(columns=["overview","genre","title"])
new_data

,tags
0,Framed in the 1940s for the double murder of h...
1,"Raj is a rich, carefree, happy-go-lucky second..."
2,"Spanning the years 1945 to 1955, a chronicle o..."
3,The true story of how businessman Oskar Schind...
4,In the continuing saga of the Corleone crime f...
...,...
9995,"The story follows the adventures of Aang, a yo..."
9996,The sharks take bite out of the East Coast whe...
9997,"During World War II, a brave, patriotic Americ..."
9998,A man named Farmer sets out to rescue his kidn...


In [16]:
#converting the text into feature vectors
cv = CountVectorizer(max_features=10000, stop_words="english")
cv

CountVectorizer(max_features=10000, stop_words='english')

In [17]:
# converting in array form
vector=cv.fit_transform(new_data["tags"].values.astype("U")).toarray()
vector

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [18]:
# number of Rows and Columns
vector.shape

(10000, 10000)

In [19]:
# getting the similarity using using cosine similarity

similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.05555556, 0.08006408, ..., 0.0729325 , 0.06804138,
        0.0347524 ],
       [0.05555556, 1.        , 0.08006408, ..., 0.        , 0.        ,
        0.        ],
       [0.08006408, 0.08006408, 1.        , ..., 0.02335709, 0.03268602,
        0.        ],
       ...,
       [0.0729325 , 0.        , 0.02335709, ..., 1.        , 0.02977457,
        0.03041495],
       [0.06804138, 0.        , 0.03268602, ..., 0.02977457, 1.        ,
        0.04256283],
       [0.0347524 , 0.        , 0.        , ..., 0.03041495, 0.04256283,
        1.        ]])

In [20]:
# number of Rows and Columns
similarity.shape

(10000, 10000)

In [21]:
# getting the movie name from the user
movie_name =input("Enter your favourite movie name: ")

In [22]:
# make a list with all the movies name given in the dataset
list_of_all_titles=movies["title"].tolist()
list_of_all_titles

['The Shawshank Redemption',
 'Dilwale Dulhania Le Jayenge',
 'The Godfather',
 "Schindler's List",
 'The Godfather: Part II',
 'Impossible Things',
 'Spirited Away',
 'Your Eyes Tell',
 'Dou kyu sei – Classmates',
 'Your Name.',
 '12 Angry Men',
 "Gabriel's Inferno",
 'Parasite',
 'The Green Mile',
 "Gabriel's Inferno: Part II",
 'The Dark Knight',
 'The Good, the Bad and the Ugly',
 'Pulp Fiction',
 'The Lord of the Rings: The Return of the King',
 "Gabriel's Inferno: Part III",
 'Forrest Gump',
 'Cinema Paradiso',
 'Seven Samurai',
 'GoodFellas',
 'Violet Evergarden: The Movie',
 'Life Is Beautiful',
 'Once Upon a Time in America',
 'Harakiri',
 'Psycho',
 'Josee, the Tiger and the Fish',
 "A Dog's Will",
 'Grave of the Fireflies',
 "One Flew Over the Cuckoo's Nest",
 'Fight Club',
 'Evangelion: 3.0+1.0 Thrice Upon a Time',
 'Spider-Man: Into the Spider-Verse',
 'City of God',
 'Hope',
 'A Silent Voice: The Movie',
 'Hotarubi no Mori e',
 "Howl's Moving Castle",
 'Neon Genesis Evang

In [23]:
# finding the close match for the movie name given by the user
related_movie=difflib.get_close_matches(movie_name,list_of_all_titles)
related_movie


['Iron Man', 'Iron Man 3', 'Iron Man 2']

In [24]:
# matched with the movies name given in the user
matched = related_movie[0]
matched

'Iron Man'

In [25]:
# finding the index of the movie name
index_of_movie = movies[movies.title==matched].index[0]
index_of_movie

969

In [26]:
# getting the list of similar movies
similarity_score = list(enumerate(similarity[index_of_movie]))
similarity_score

[(0, 0.0),
 (1, 0.0),
 (2, 0.0),
 (3, 0.0),
 (4, 0.0),
 (5, 0.0625),
 (6, 0.0),
 (7, 0.0),
 (8, 0.0),
 (9, 0.0),
 (10, 0.0),
 (11, 0.048112522432468816),
 (12, 0.0),
 (13, 0.042874646285627205),
 (14, 0.0),
 (15, 0.036084391824351615),
 (16, 0.051031036307982884),
 (17, 0.10425720702853739),
 (18, 0.0),
 (19, 0.0),
 (20, 0.057353933467640436),
 (21, 0.0),
 (22, 0.044194173824159216),
 (23, 0.0),
 (24, 0.0),
 (25, 0.0),
 (26, 0.0),
 (27, 0.037688918072220454),
 (28, 0.0),
 (29, 0.0),
 (30, 0.0),
 (31, 0.0),
 (32, 0.0),
 (33, 0.05),
 (34, 0.11572751247156893),
 (35, 0.18569533817705186),
 (36, 0.0),
 (37, 0.0),
 (38, 0.0),
 (39, 0.0),
 (40, 0.0),
 (41, 0.07624928516630233),
 (42, 0.11572751247156893),
 (43, 0.0),
 (44, 0.042874646285627205),
 (45, 0.03904344047215152),
 (46, 0.06933752452815364),
 (47, 0.0625),
 (48, 0.0),
 (49, 0.05892556509887897),
 (50, 0.0),
 (51, 0.0),
 (52, 0.0),
 (53, 0.0),
 (54, 0.1599005372667078),
 (55, 0.042874646285627205),
 (56, 0.0),
 (57, 0.0),
 (58, 0.069

In [27]:
# finding length
len(similarity_score)

10000

In [28]:
# sorting the movies based on their similarity score
sorted_similar_movies = sorted(similarity_score, reverse=True, key = lambda vector:vector[1])
sorted_similar_movies

[(969, 1.0),
 (3993, 0.3244428422615251),
 (3563, 0.32274861218395134),
 (9171, 0.32274861218395134),
 (4424, 0.2773500981126146),
 (1813, 0.2668724980820582),
 (6089, 0.2551551815399144),
 (1727, 0.25),
 (4749, 0.25),
 (8026, 0.25),
 (5772, 0.24253562503633297),
 (6216, 0.24253562503633297),
 (7020, 0.24253562503633297),
 (6570, 0.2405626121623441),
 (2100, 0.2401922307076307),
 (7146, 0.24019223070763068),
 (8885, 0.23570226039551587),
 (9470, 0.23570226039551587),
 (3185, 0.22941573387056174),
 (6477, 0.2282177322938192),
 (5538, 0.22613350843332275),
 (9729, 0.22613350843332272),
 (2929, 0.22360679774997896),
 (8680, 0.22360679774997896),
 (1987, 0.2182178902359924),
 (5092, 0.2182178902359924),
 (6288, 0.2175970699446223),
 (561, 0.2165063509461097),
 (6837, 0.21320071635561041),
 (9360, 0.21128856368212912),
 (7101, 0.20851441405707477),
 (9325, 0.20851441405707477),
 (962, 0.20801257358446093),
 (1405, 0.20801257358446093),
 (4890, 0.20801257358446093),
 (5891, 0.208012573584460

In [29]:
# print the name of similar movies based on the index
print("Movies suggested for you:\n")
for i in sorted_similar_movies[:10]:
    print(movies.iloc[i[0]].title)

Movies suggested for you:

Iron Man
Iron Man 2
Iron Man 3
The Lawnmower Man
Mazinger Z: Infinity
Spider-Man: Homecoming
Outlander
Justice League Dark
Ransom
The 6th Day


In [30]:
# Movies Recommandation System

movie_name =input("Enter your favourite movie name: ")

list_of_all_titles=movies["title"].tolist()

related_movie=difflib.get_close_matches(movie_name,list_of_all_titles)

matched = related_movie[0]

index_of_movie = movies[movies.title==matched].index[0]

similarity_score = list(enumerate(similarity[index_of_movie]))

sorted_similar_movies = sorted(similarity_score, reverse=True, key = lambda vector:vector[1])

print("Movies suggested for you:\n")
for i in sorted_similar_movies[:10]:
    print(movies.iloc[i[0]].title)

Movies suggested for you:

Iron Man
Iron Man 2
Iron Man 3
The Lawnmower Man
Mazinger Z: Infinity
Spider-Man: Homecoming
Outlander
Justice League Dark
Ransom
The 6th Day
